# 🤖 Notebook 03 — Agent Demo

**Goal:** Demonstrate the full multi-agent system with 5 scenario types:

| Scenario | Expected Agent | Description |
|----------|---------------|-------------|
| General Q&A | RAG agent | Evidence-based medical info |
| Symptom triage | Symptom agent | Urgency assessment |
| Drug info | Drug agent | Medication details |
| Lifestyle | Lifestyle agent | Prevention & wellness |
| Emergency | Emergency bypass | Instant response (<10ms) |

**Prerequisites:** Run `02_rag_setup.ipynb` first to build the ChromaDB store.

In [ ]:
# Cell 1: Setup & imports
import sys, time, uuid
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

# Load env vars from .env (GROQ_API_KEY etc.)
from dotenv import load_dotenv
load_dotenv('../.env')

from langchain_core.messages import HumanMessage
from src.agents.graph import build_medical_graph
from src.utils import get_logger

logger = get_logger('03_demo')
print('✅ Imports OK')

In [ ]:
# Cell 2: Build the medical agent graph (with in-memory MemorySaver)
graph = build_medical_graph()
print('✅ Medical agent graph compiled')

In [ ]:
# Cell 3: Helper — pretty print agent response
def run_agent(query: str, session_id: str | None = None) -> dict:
    """Invoke the graph and print a formatted response."""
    if session_id is None:
        session_id = str(uuid.uuid4())
    
    config = {'configurable': {'thread_id': session_id}}
    t0 = time.time()
    result = graph.invoke({'messages': [HumanMessage(content=query)]}, config=config)
    latency_ms = (time.time() - t0) * 1000
    
    route = result.get('route_to', 'unknown')
    urgency = result.get('urgency_level', '-')
    is_emergency = result.get('is_emergency', False)
    response = result.get('agent_response', '')
    
    print('=' * 70)
    print(f'🔵 QUERY:     {query}')
    print(f'🔀 ROUTE:     {route}')
    print(f'⚡ URGENCY:   {urgency}/5' + (' 🚨 EMERGENCY' if is_emergency else ''))
    print(f'⏱️ LATENCY:   {latency_ms:.0f} ms')
    print(f'\n📝 RESPONSE:\n{response}')
    print('=' * 70)
    return result

In [ ]:
# Cell 4: DEMO CASE 1 — General Medical Q&A (RAG agent)
result1 = run_agent('What is Type 2 Diabetes and what causes it?', 'demo-rag-01')

In [ ]:
# Cell 5: DEMO CASE 2 — Symptom Assessment (Symptom agent)
result2 = run_agent(
    'I have been experiencing frequent urination, excessive thirst, and blurred vision for two weeks.',
    'demo-symptom-01'
)

In [ ]:
# Cell 6: DEMO CASE 3 — Drug Information (Drug agent)
result3 = run_agent('What are the side effects and contraindications of metformin?', 'demo-drug-01')

In [ ]:
# Cell 7: DEMO CASE 4 — Lifestyle & Prevention (Lifestyle agent)
result4 = run_agent(
    'What dietary changes and exercise habits can help manage hypertension naturally?',
    'demo-lifestyle-01'
)

In [ ]:
# Cell 8: DEMO CASE 5 — Emergency bypass (no LLM call)
t0 = time.time()
result5 = run_agent('I have severe chest pain radiating to my left arm!', 'demo-emergency-01')
print(f'\n⚡ Emergency bypass latency: {(time.time()-t0)*1000:.1f} ms (should be <50ms)')

In [ ]:
# Cell 9: DEMO CASE 6 — Multi-turn conversation memory
SESSION = 'demo-memory-01'
print('--- Turn 1 ---')
run_agent('Can you tell me about hypertension?', SESSION)
print()
print('--- Turn 2 (should remember previous context) ---')
run_agent('What foods should I avoid given what we just discussed?', SESSION)

In [ ]:
# Cell 10: Routing summary across all demo cases
import pandas as pd

cases = [
    ('General Q&A', result1),
    ('Symptom triage', result2),
    ('Drug info', result3),
    ('Lifestyle', result4),
    ('Emergency', result5),
]

summary = pd.DataFrame([
    {
        'Case': name,
        'Route': r.get('route_to', 'N/A'),
        'Urgency': r.get('urgency_level', '-'),
        'Emergency': '🚨 YES' if r.get('is_emergency') else 'No',
        'Response Chars': len(r.get('agent_response', '')),
    }
    for name, r in cases
])
print(summary.to_string(index=False))